# Metatlas2 Standalone Development Workflow

Execute cells sequentially to run the complete metatlas2 pipeline using production code.

In [ ]:
import os
import logging
from pathlib import Path
import metatlas2.workflows as wfs
import metatlas2.load_tools as ldt
import metatlas2.logging_config as lcf
from metatlas2.add_compounds_to_db import add_compounds_to_db
from metatlas2.add_atlases_to_db import add_atlases_to_db
from metatlas2.run_targeted_analysis import set_up_paths

lcf.setup_logging(log_level=logging.INFO, log_to_stdout=True)
logger = lcf.get_logger("standalone_workflow")

In [ ]:
project_name = "20260101_JGI_XX_000000_STANDALONE-DEV_test_EXP000_HILICZ_TESTXXXX"
rt_alignment_number = 0
analysis_number = 0

In [ ]:
logger.info(f"Starting workflow for project: {project_name} with RT alignment number: {rt_alignment_number} and analysis number: {analysis_number}")

In [ ]:
logger.info("Setting up data directory...")
DATA_DIR = Path(os.environ.get('METATLAS_DATA_DIR', Path.home() / '.metatlas2-dev'))
os.environ['METATLAS_DATA_DIR'] = str(DATA_DIR)

In [ ]:
logger.info("Adding compounds to database...")
add_compounds_to_db(config_path=str(DATA_DIR / "configs" / "compounds_config.yaml"), overwrite_db=True)
logger.info("Compounds added successfully")

In [ ]:
logger.info("Creating atlases...")
add_atlases_to_db(config_path=str(DATA_DIR / "configs" / "atlases_config.yaml"))
logger.info("Atlases created successfully.")

In [ ]:
logger.info("Loading analysis configuration file and setting up project output paths...")
config = ldt.load_metatlas2_config(str(DATA_DIR / "configs" / "analysis_config.yaml"))
paths = set_up_paths(
    config=config,
    project_name=project_name,
    rt_alignment_number=rt_alignment_number,
    analysis_number=analysis_number,
)

In [ ]:
logger.info("Resetting logging to notebook output and log file...")
lcf.setup_logging(log_level=logging.INFO, log_file=paths["log_path"], log_to_stdout=True)
logger = lcf.get_logger("standalone_workflow")
logger.info("Kicking off project workflow...")

In [ ]:
logger.info("Updating config with atlas UIDs...")

# Update RT alignment atlas UID
qc_atlas_uid = ldt.parse_atlas_creator_log( f"{str(DATA_DIR)}/configs/add_atlases_to_db.log", "HILICZ", "POS", "QC", "MAIN")
config.rt_alignment_config['HILICZ']['ATLAS']['uid'] = qc_atlas_uid
logger.info(f"Updated RT alignment atlas UID: {qc_atlas_uid}")

# Update targeted analysis atlas UID
for ta in config.targeted_analyses:
    atlas_uid = ldt.parse_atlas_creator_log(
        f"{str(DATA_DIR)}/configs/add_atlases_to_db.log",
        ta.chromatography,
        ta.polarity,
        ta.analysis_type,
        ta.analysis_name
    )
    ta.atlas_uid = atlas_uid
    logger.info(f"Updated targeted analysis atlas UID for {ta.chromatography}-{ta.polarity}-{ta.analysis_type}-{ta.analysis_name}: {atlas_uid}")

In [ ]:
logger.info("Running project setup...")
wfs.run_project_setup(
    project_name=project_name,
    config=config,
    paths=paths,
    overwrite_existing=True,
    rt_alignment_number=rt_alignment_number,
    analysis_number=analysis_number,
)
logger.info("Project setup complete")

In [ ]:
logger.info("Running RT alignment...")
wfs.run_rt_alignment(
    project_name=project_name,
    rt_alignment_number=rt_alignment_number,
    analysis_number=analysis_number,
)
logger.info("RT alignment complete")

In [ ]:
logger.info("Running auto-identification...")
wfs.run_auto_identification(
    project_name=project_name,
    rt_alignment_number=rt_alignment_number,
    analysis_number=analysis_number,
    image_tag="latest",
)
logger.info("Auto-identification complete. Open the generated GUI notebooks to curate.")